# Wine Quality Prediction

**Oasis Infobyte Data Analytics Internship — Level 2, Task 2**

This notebook predicts wine quality classes from physicochemical measurements and compares Random Forest, SGD and Support Vector Classifier models.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the UCI Wine Quality Dataset


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

wine = pd.read_csv(url, sep=";")

print("Dataset shape:", wine.shape)
display(wine.head())


## 3. Inspect Structure and Original Quality Distribution


In [ ]:
print("Data types:")
display(wine.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(wine.isna().sum().to_frame("missing_values"))

print("\nDescriptive statistics:")
display(wine.describe().T)

print("\nOriginal quality distribution:")
quality_counts = wine["quality"].value_counts().sort_index()
display(quality_counts.to_frame("count"))

plt.figure(figsize=(9, 5))
sns.countplot(data=wine, x="quality")
plt.title("Original Wine Quality Score Distribution")
plt.xlabel("Quality Score")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 4. Exploratory Data Analysis — Chemical Features


In [ ]:
chemical_features = [col for col in wine.columns if col != "quality"]

for feature in chemical_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(wine[feature], kde=True)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


## 5. Correlation Heatmap


In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(
    wine.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("Correlation with original quality:")
display(
    wine.corr(numeric_only=True)["quality"]
    .sort_values(ascending=False)
    .to_frame("correlation")
)


## 6. Feature Engineering — Three Quality Classes

The original quality score is converted into three groups:

- **Low Quality:** score <= 5
- **Medium Quality:** score == 6
- **High Quality:** score >= 7

This transformation turns the original ordinal score into a multiclass classification target while keeping the grouping easy to explain.

Because the high-quality category may contain fewer observations, the resulting class distribution is checked before training.


In [ ]:
def quality_group(score):
    if score <= 5:
        return "Low"
    elif score == 6:
        return "Medium"
    return "High"

wine["quality_class"] = wine["quality"].apply(quality_group)

class_counts = wine["quality_class"].value_counts()
class_percentages = wine["quality_class"].value_counts(normalize=True).mul(100).round(2)

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages
})

display(class_distribution)

plt.figure(figsize=(8, 5))
sns.countplot(
    data=wine,
    x="quality_class",
    order=["Low", "Medium", "High"]
)
plt.title("Three-Class Wine Quality Distribution")
plt.xlabel("Quality Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## Class Imbalance Discussion

The class distribution above should be considered before interpreting model accuracy.

If one class has substantially more observations than another, a classifier can achieve relatively high accuracy while still performing poorly on the minority class.

For this reason, the classification reports and confusion matrices are examined alongside accuracy rather than relying on accuracy alone.


## 7. Prepare Features and Stratified Train/Test Split


In [ ]:
X = wine.drop(columns=["quality", "quality_class"])
y = wine["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).mul(100).round(2).to_frame("percentage"))

print("\nTesting class proportions:")
display(y_test.value_counts(normalize=True).mul(100).round(2).to_frame("percentage"))


## 8. Train Three Classification Models

The required classifiers are:

1. Random Forest
2. Stochastic Gradient Descent
3. Support Vector Classifier

Feature standardisation is used for SGD and SVC because these models are sensitive to feature scale. Random Forest does not require scaling, so it uses the raw features.


In [ ]:
models = {
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        ))
    ]),
    "SGD": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SGDClassifier(
            random_state=42,
            max_iter=2000,
            tol=1e-3,
            class_weight="balanced"
        ))
    ]),
    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=42
        ))
    ])
]

predictions = {}
reports = {}
metrics = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred

    accuracy = accuracy_score(y_test, pred)
    report = classification_report(y_test, pred, output_dict=True)
    reports[name] = report

    metrics.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro Precision": report["macro avg"]["precision"],
        "Macro Recall": report["macro avg"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"]
    })

metrics_df = pd.DataFrame(metrics).sort_values("Macro F1", ascending=False)

display(metrics_df.round(4))


## 9. Classification Reports


In [ ]:
for name in models:
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(y_test, predictions[name], digits=4))


## 10. Confusion Matrices


In [ ]:
for name in models:
    cm = confusion_matrix(
        y_test,
        predictions[name],
        labels=["Low", "Medium", "High"]
    )

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=["Low", "Medium", "High"],
        yticklabels=["Low", "Medium", "High"]
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


## 11. Random Forest Feature Importance


In [ ]:
rf_model = models["Random Forest"].named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(feature_importance)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 12. Model Comparison

The table below compares all three required classifiers using accuracy and macro-averaged precision, recall and F1-score.

Macro F1 is especially useful when class sizes differ because it gives each class equal weight.


In [ ]:
comparison = metrics_df.reset_index(drop=True)

display(comparison.round(4))

comparison.to_csv(
    "wine_quality_model_comparison.csv",
    index=False
)

feature_importance.to_csv(
    "random_forest_feature_importance.csv",
    index=False
)


## 13. Conclusion

Model suitability should be judged using the evaluation results generated above.

Accuracy alone should not determine the choice when the quality classes are imbalanced. Macro precision, macro recall, macro F1 and the confusion matrix provide additional information about how consistently each model handles Low, Medium and High quality wines.

The model with the strongest overall validation metrics can be considered the most suitable candidate for this dataset, subject to further validation, operational requirements and the cost of different classification errors before real deployment.


## 14. Final Checklist


In [ ]:
checklist = {
    "Dataset loaded and inspected": True,
    "Class distribution analysed": True,
    "Chemical feature EDA": True,
    "Correlation heatmap": True,
    "Class imbalance discussed": True,
    "Three-class target engineered": True,
    "Stratified train/test split": True,
    "Random Forest trained": True,
    "SGD trained": True,
    "SVC trained": True,
    "Accuracy and classification reports": True,
    "Confusion matrices": True,
    "Random Forest feature importance": True,
    "Model comparison table": True
}

display(pd.DataFrame(
    {"Completed": checklist.values()},
    index=checklist.keys()
))
